## Fetch trades and quotes for stock and one option

In [1]:
if False:
    import os
    os.environ["QUANTDATA_CACHE_MODE"] = "off" # turns off global cache mode ... good for testing but not in general

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.massive.client
%aimport quantdata.massive.options.contracts
%aimport quantdata.massive.processing

from datetime import date, datetime
import warnings
from quantdata.massive.client import MassiveClient
from cdxcore.util import fmt_date, fmt_dict, to_date
from cdxcore.verbose import Context

import polars as pl

def test():

    ticker = "SPY"
    d      = to_date("2024-06-17")
    client = MassiveClient()
    option_ticker = "O:SPY240624C005240"

    def package( ticker, verbose = Context.all ):

        with verbose.write_t(f"Fetching OCV, trades, quotes for {ticker} on {fmt_date(d)}... ", end='') as tme:
            ocv = client.fetch_day_summary(ticker, d, adjusted=True)
            verbose.write(f"done: found {fmt_dict(ocv)}. This took {tme}.", head=False)

        with verbose.write_t(f"Fetching trades for {ticker} on {fmt_date(d)}... ", end='') as tme:
            trades = client.fetch_trades(ticker, d) # extracts trades, removed zero size and price, and excludes trades with NORMAL_TRADE_EXCLUDE_FLAGS
            verbose.write(f"done: found {len(trades)} trades. This took {tme}.", head=False)

        with verbose.write_t(f"Fetching quotes for {ticker} on {fmt_date(d)}... ", end='') as tme:
            quotes = client.fetch_quotes(ticker, d)
            verbose.write(f"done: found {len(quotes)} quotes. This took {tme}.", head=False)

        return ocv, trades, quotes  

    package(ticker)
    ocv, trades, quotes = package(option_ticker)

    display(ocv)
    display(trades.head())
    display(quotes.head())

test()



00: Fetching OCV, trades, quotes for SPY on 2024-06-17... done: found open: 542.08, close: 547.1, volume: 55839457.0, high: 548.53, low: 541.6072, afterHours: 547.54, preMarket: 542.55 and adjusted: True. This took 0s.
00: Fetching trades for SPY on 2024-06-17... done: found 402271 trades. This took 0.015s.
00: Fetching quotes for SPY on 2024-06-17... done: found 1957755 quotes. This took 0.013s.
00: Fetching OCV, trades, quotes for O:SPY240624C005240 on 2024-06-17... 

HTTPError: No open/close/volume for O:SPY240624C005240 on 2024-06-17 url https://api.massive.com/v1/open-close/O:SPY240624C005240/2024-06-17: {"status":"NOT_FOUND","request_id":"3c1b36aa56f1918d02e03bf7bb590546","message":"Data not found."}

In [3]:
d      = to_date("2024-06-17")
client = MassiveClient()
option_ticker = "O:SPY240624C00524000"

client.has_any_quotes_or_trades(option_ticker, d)





(True, False)